In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content/drive/MyDrive/data

/content/drive/MyDrive/data


In [3]:
! ls

samsum-test.csv  samsum-train.csv  samsum-validation.csv


In [4]:
! nvidia-smi

Wed Jun 18 08:55:36 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
! pip install --upgrade accelerate

In [6]:
! pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [9]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
from datasets import load_dataset
import pandas as pd
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device

device(type='cuda')

In [11]:
from transformers import AutoModel, AutoTokenizer

AutoModel.from_pretrained("google/pegasus-cnn_dailymail")
AutoTokenizer.from_pretrained("google/pegasus-cnn_dailymail")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of PegasusModel were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['decoder.embed_positions.weight', 'encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PegasusTokenizerFast(name_or_path='google/pegasus-cnn_dailymail', vocab_size=96103, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'mask_token': '<mask_2>', 'additional_special_tokens': ['<mask_1>', '<unk_2>', '<unk_3>', '<unk_4>', '<unk_5>', '<unk_6>', '<unk_7>', '<unk_8>', '<unk_9>', '<unk_10>', '<unk_11>', '<unk_12>', '<unk_13>', '<unk_14>', '<unk_15>', '<unk_16>', '<unk_17>', '<unk_18>', '<unk_19>', '<unk_20>', '<unk_21>', '<unk_22>', '<unk_23>', '<unk_24>', '<unk_25>', '<unk_26>', '<unk_27>', '<unk_28>', '<unk_29>', '<unk_30>', '<unk_31>', '<unk_32>', '<unk_33>', '<unk_34>', '<unk_35>', '<unk_36>', '<unk_37>', '<unk_38>', '<unk_39>', '<unk_40>', '<unk_41>', '<unk_42>', '<unk_43>', '<unk_44>', '<unk_45>', '<unk_46>', '<unk_47>', '<unk_48>', '<unk_49>', '<unk_50>', '<unk_51>', '<unk_52>', '<unk_53>', '<unk_54>', '<unk_55>', '<unk_56>', '<unk_57>', '<unk_58>', '<unk_5

In [12]:
model_ckpt = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [13]:
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
dataset = load_dataset("knkarthick/samsum")

README.md:   0%|          | 0.00/4.36k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/504k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/522k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14732 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

In [16]:
dataset["train"]["dialogue"][1]

'Olivia: Who are you voting for in this election? \nOliver: Liberals as always.\nOlivia: Me too!!\nOliver: Great'

In [17]:
dataset["train"][1]["summary"]

'Olivia and Olivier are voting for liberals in this election. '

In [18]:
split_lengths = [len(dataset[split]) for split in dataset]

print(f"Spllit lengths: {split_lengths}")
print(f"Features: {dataset['train'].column_names}")
print("\nDialogue: ")

print(dataset["test"][1]["dialogue"])

print("\nSummary: ")

print(dataset["test"][1]["summary"])

Spllit lengths: [14732, 818, 819]
Features: ['id', 'dialogue', 'summary']

Dialogue: 
Eric: MACHINE!
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: Sure.
Rob: Turns out no! There are some of his stand-ups on youtube.
Eric: Gr8! I'll watch them now!
Rob: Me too!
Eric: MACHINE!
Rob: MACHINE!
Eric: TTYL?
Rob: Sure :)

Summary: 
Eric and Rob are going to watch a stand-up on youtube.


In [24]:
def convert_examples_to_features(batch):
    inputs = ["summarize: " + d if d is not None else "summarize: " for d in batch["dialogue"]]
    targets = [t if t is not None else "" for t in batch["summary"]]

    model_inputs = tokenizer(
        inputs,
        padding="max_length",
        truncation=True,
        max_length=512,
    )

    labels = tokenizer(
        targets,
        padding="max_length",
        truncation=True,
        max_length=128,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [25]:
dataset_samsum_pt = dataset.map(convert_examples_to_features, batched=True)

Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [26]:
dataset_samsum_pt["train"]

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 14732
})

In [28]:
dataset_samsum_pt["train"]["input_ids"][2]

[24710,
 151,
 4776,
 151,
 4451,
 108,
 180,
 131,
 116,
 164,
 152,
 5377,
 151,
 6843,
 4301,
 83678,
 108,
 125,
 140,
 313,
 112,
 171,
 1425,
 113,
 1549,
 155,
 2371,
 164,
 64428,
 4776,
 151,
 463,
 368,
 119,
 511,
 124,
 557,
 152,
 5377,
 151,
 4384,
 119,
 235,
 108,
 18857,
 1549,
 111,
 1596,
 2073,
 56616,
 161,
 418,
 5377,
 151,
 3183,
 3469,
 125,
 131,
 267,
 696,
 161,
 130,
 116,
 111,
 171,
 579,
 5377,
 151,
 184,
 195,
 313,
 112,
 38244,
 114,
 5713,
 167,
 1088,
 113,
 1553,
 125,
 131,
 267,
 1461,
 181,
 38244,
 316,
 9609,
 4776,
 151,
 321,
 557,
 1549,
 125,
 1253,
 881,
 93882,
 3111,
 241,
 4911,
 207,
 5033,
 118,
 557,
 18576,
 4776,
 151,
 168,
 288,
 1107,
 5377,
 151,
 1516,
 108,
 1556,
 125,
 131,
 267,
 171,
 120,
 4776,
 151,
 125,
 163,
 172,
 303,
 450,
 121,
 12397,
 115,
 9994,
 11485,
 669,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0

In [29]:
dataset_samsum_pt["train"]["attention_mask"][1]

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,


In [30]:
from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model= model_pegasus)

In [34]:
! pip install --upgrade transformers


In [36]:
from transformers import TrainingArguments, Trainer

trainer_args = TrainingArguments(
    output_dir='pegasus-samsum',
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    weight_decay=0.01,
    logging_steps=10,
    eval_steps=500,
    save_steps=int(1e6),  # fix lỗi kiểu dữ liệu
    gradient_accumulation_steps=16
)

In [39]:
dataset_samsum_pt

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14732
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})

In [40]:
trainer = Trainer(
    model = model_pegasus,
    args = trainer_args,
    tokenizer = tokenizer,
    data_collator=seq2seq_data_collator,
    train_dataset=dataset_samsum_pt["test"],
    eval_dataset=dataset_samsum_pt["validation"]
)

<ipython-input-40-4242671842>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [41]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 286ngocson (286ngocson-nvidia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,10.589500
20,10.677100
30,10.505900
40,10.508400
50,10.354700


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 128, 'min_length': 32, 'num_beams': 8, 'length_penalty': 0.8}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=52, training_loss=10.360572814941406, metrics={'train_runtime': 589.2248, 'train_samples_per_second': 1.39, 'train_steps_per_second': 0.088, 'total_flos': 1183235677618176.0, 'train_loss': 10.360572814941406, 'epoch': 1.0})

In [42]:
def generate_batch_sized_chunks(list_of_elements, batch_size):
    """
    Split the dataset into smaller batches that we can process simultaneously
    Yield successive batch-sized chunks from list_of_elements.
    """
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]

def calculate_metric_on_test_ds(dataset, metric, model, tokenizer, batch_size=16, device=device, column_text='article', column_summary='highlights'):
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    for article_batch, target_batch in tqdm(zip(article_batches, target_batches), total=len(article_batches)):
        inputs = tokenizer(article_batch, max_length=1024, truncation=True, padding="max_length", return_tensors="pt")
        summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                                   attention_mask=inputs["attention_mask"].to(device),
                                   length_penalty=0.8, num_beams=8, max_length=128)
        """parameter for length penalty ensures that model does not generate sequen"""
        decode_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                             clean_up_tokenization_spaces=True) for s in summaries]
        decode_summaries = [d.replace("", " ") for d in decoded_summaries]

        metric.add_batch(predictions=decode_summaries, references=target_batch)

    score = metric.compute()
    return score

In [45]:
!pip install evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.9 MB/s eta 0:00:00


In [46]:
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
import evaluate

metric = evaluate.load("rouge")  # hoặc "bleu", "accuracy", v.v.
#rouge_metric = load_metric('rouge')

In [50]:
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

# Hàm tính ROUGE thủ công (không dùng Trainer)
def calculate_metric_on_test_ds(dataset, tokenizer, model, batch_size=2, column_text="dialogue", column_summary="summary"):
    inputs = [f"summarize: {x}" for x in dataset[column_text]]
    references = dataset[column_summary]

    predictions = []
    for i in range(0, len(inputs), batch_size):
        batch = inputs[i:i+batch_size]
        inputs_tokenized = tokenizer(batch, truncation=True, padding=True, return_tensors="pt").to(model.device)

        with torch.no_grad():
            summaries_ids = model.generate(
                input_ids=inputs_tokenized["input_ids"],
                attention_mask=inputs_tokenized["attention_mask"],
                max_length=60,
                num_beams=4,
                length_penalty=2.0,
                early_stopping=True,
            )
        decoded_summaries = tokenizer.batch_decode(summaries_ids, skip_special_tokens=True)
        predictions.extend(decoded_summaries)

    results = metric.compute(predictions=predictions, references=references)
    return results

# Gọi tính điểm
score = calculate_metric_on_test_ds(dataset["test"].select(range(10)), tokenizer=tokenizer, model=trainer.model)

# Lấy điểm f1 (fmeasure) cho từng loại ROUGE
rouge_dict = {k: round(v, 4) for k, v in score.items() if k in rouge_names}

# Hiển thị bằng pandas
pd.DataFrame(rouge_dict, index=["pegasus"])

,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.2777,0.0617,0.2112,0.2119


In [51]:
# Save
model_pegasus.save_pretrained("pegasus-samsum-model")
# Save tokenizer
tokenizer.save_pretrained("tokenizer")

('tokenizer/tokenizer_config.json',
 'tokenizer/special_tokens_map.json',
 'tokenizer/spiece.model',
 'tokenizer/added_tokens.json',
 'tokenizer/tokenizer.json')

In [56]:
%cd /content/drive/MyDrive/data

/content/drive/MyDrive/data


In [57]:
!ls tokenizer


ls: cannot access '/content/tokenizer': No such file or directory


In [58]:
tokenizer = AutoTokenizer.from_pretrained("tokenizer")

In [60]:
gen_kwargs = {"length_penalty": 0.8, "num_beams": 8, "max_length":128}

sample_text = dataset["test"][0]["dialogue"]
reference = dataset["test"][0]["summary"]

pipe = pipeline("summarization", model="pegasus-samsum-model", tokenizer=tokenizer)

print("Dialogue:")
print(sample_text)

print("\nReference Summary:")
print(reference)

print("\nModel Summary:")
print(pipe(sample_text, **gen_kwargs)[0]["summary_text"])

Device set to use cuda:0
Your max_length is set to 128, but your input_length is only 122. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=61)


Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Reference Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

Model Summary:
Amanda: Ask Larry Amanda: He called her last time we were at the park together .<n>Hannah: I'd rather you texted him .<n>Amanda: Just text him .


In [61]:
!git config --global user.email "286sunsun@gmail.com"

In [62]:
!git config --global user.name "NgocSon-AI"

In [63]:
!git clone https://github.com/NgocSon-AI/GenerativeAIMateryFullCourse.git

Cloning into 'GenerativeAIMateryFullCourse'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 60 (delta 10), reused 53 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 29.08 MiB | 12.90 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Updating files: 100% (29/29), done.


In [64]:
%cd GenerativeAIMateryFullCourse/

/content/drive/MyDrive/data/GenerativeAIMateryFullCourse


In [65]:
!cp SummarizerTextProject.ipynb

cp: missing destination file operand after 'SummarizerTextProject.ipynb'
Try 'cp --help' for more information.
